[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-11-custom-tools.ipynb#scrollTo=a1b2c3d4)

---
# Day 11 · Custom Tools — @tool, StructuredTool, and DuckDB Integration
**certified-journeys / llm-engineering-certified** · Day 11 · Tool Engineering

> **Goal for today:** Build custom LangChain tools using `@tool` and `StructuredTool`, wire them into an AgentExecutor, integrate DuckDB as a data source, and handle tool failures gracefully.

In [ ]:
%pip install -q langchain langchain-openai langchain-community wikipedia duckdb pydantic

## Step 1 · The `@tool` Decorator — Simplest Custom Tool

The `@tool` decorator turns any Python function into a LangChain tool. The **function docstring becomes the tool description** — the agent reads it verbatim to decide when to call the tool.

| Decorator | Best for | Schema |
|-----------|----------|--------|
| `@tool` | Simple, single-arg functions | Auto-inferred from type hints |
| `StructuredTool` | Multi-arg tools with Pydantic schema | Explicit Pydantic model |
| `Tool` (legacy) | Single string input, backward compat | Single `str` arg only |

Rules for good tool descriptions:
- Be specific about **when** to use the tool
- Name the input format explicitly (`city name as a string`)
- Avoid vague words like "useful" or "helpful"

In [ ]:
import wikipedia
from langchain.tools import tool

@tool
def get_wikipedia_summary(query: str) -> str:
    """Fetch a concise Wikipedia summary for a given topic or person.
    Use this tool whenever the user asks about a factual topic, historical event,
    scientific concept, or well-known person. Input: the search query as a plain string.
    Returns the first paragraph of the Wikipedia article or an error message."""
    try:
        # wikipedia.summary raises DisambiguationError or PageError on bad queries
        return wikipedia.summary(query, sentences=3, auto_suggest=True)
    except wikipedia.exceptions.DisambiguationError as e:
        # Return the first candidate so the agent can refine its query
        return f"Ambiguous query. Suggestions: {', '.join(e.options[:5])}"
    except wikipedia.exceptions.PageError:
        return f"No Wikipedia page found for '{query}'."
    except Exception as e:
        return f"Wikipedia lookup failed: {str(e)}"

# Inspect what LangChain sees
print("Tool name:", get_wikipedia_summary.name)
print("Tool description:", get_wikipedia_summary.description[:120])
print("Args schema:", get_wikipedia_summary.args)

**What just happened?**
- `@tool` extracted the function name (`get_wikipedia_summary`) as the tool name
- The **docstring** became the description — this is exactly what the LLM reads to decide whether to call this tool
- Type hint `query: str` was auto-converted to a JSON schema with a single `query` field
- The `try/except` ensures tool failures return a string, never raise — **agents cannot recover from bare exceptions**

## Step 2 · `StructuredTool` with Pydantic Schema — Multi-Argument Tools

`StructuredTool` is the right choice when your tool takes **multiple typed arguments**. You define an explicit Pydantic model for the args schema, which:
- Gives the LLM a clear contract (field names, types, descriptions)
- Enables validation before your function runs
- Makes error messages meaningful when the agent passes bad args

The production equivalent of this weather stub would call the OpenWeatherMap API — we use a local mock here.

In [ ]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field
import json

# --- Pydantic schema defines the tool's input contract ---
class WeatherInput(BaseModel):
    city: str = Field(..., description="The city name, e.g. 'London' or 'Tokyo'")
    units: str = Field(default="metric", description="Temperature units: 'metric' (Celsius) or 'imperial' (Fahrenheit)")

# --- Mock implementation (production: call OpenWeatherMap API) ---
MOCK_WEATHER = {
    "london":  {"metric": 14, "imperial": 57, "desc": "Partly cloudy"},
    "tokyo":   {"metric": 22, "imperial": 72, "desc": "Clear skies"},
    "new york": {"metric": 18, "imperial": 64, "desc": "Windy"},
    "paris":   {"metric": 16, "imperial": 61, "desc": "Overcast"},
}

def _get_weather(city: str, units: str = "metric") -> str:
    """Internal function — StructuredTool wraps this."""
    key = city.lower()
    unit_symbol = "°C" if units == "metric" else "°F"
    if key not in MOCK_WEATHER:
        return f"Weather data unavailable for '{city}'. Try a major city name."
    data = MOCK_WEATHER[key]
    temp = data[units] if units in data else data["metric"]
    return f"{city.title()}: {data['desc']}, {temp}{unit_symbol}"

# Wrap with StructuredTool — description tells the agent exactly when to call it
weather_tool = StructuredTool.from_function(
    func=_get_weather,
    name="get_weather",
    description=(
        "Get the current weather for a specific city. "
        "Use this when the user asks about weather, temperature, or forecast in a named city. "
        "Requires 'city' (string). Optionally specify 'units' as 'metric' or 'imperial'."
    ),
    args_schema=WeatherInput,
)

# Smoke-test the tool directly (bypassing the agent)
print(weather_tool.run({"city": "Tokyo", "units": "metric"}))
print(weather_tool.run({"city": "London", "units": "imperial"}))
print("Args schema fields:", list(weather_tool.args.keys()))

**What just happened?**
- `WeatherInput` gave the tool a **formal JSON schema** — LangChain serializes this and puts it in the LLM's system prompt
- `StructuredTool.from_function` separated the schema from the implementation — the function stays clean
- `Field(description=...)` annotations flow through to the schema; more specific field descriptions → better argument extraction by the LLM
- **Mock data** keeps the code runnable in Colab — swap `_get_weather` for a real API call with one line change

## Step 3 · DuckDB Tool — Query Local Data

DuckDB is the local alternative to BigQuery or Snowflake: in-process, zero-config, SQL-compatible. A DuckDB tool lets your agent query structured data without any cloud dependency.

**Production equivalent:** Replace the DuckDB file path with a connection to your warehouse (BigQuery, Redshift, Snowflake) using the same tool interface — the agent doesn't need to know the difference.

Key design decision: **return a formatted string**, not a raw result object — agents can only process strings.

In [ ]:
import duckdb

# --- Create an in-memory DuckDB with sample sales data ---
conn = duckdb.connect()  # in-memory; use duckdb.connect('mydb.duckdb') for persistent

conn.execute("""
    CREATE TABLE sales AS
    SELECT * FROM (VALUES
        ('2024-01', 'Electronics', 45200.00),
        ('2024-01', 'Clothing',    12300.00),
        ('2024-02', 'Electronics', 51000.00),
        ('2024-02', 'Clothing',    14500.00),
        ('2024-03', 'Electronics', 48700.00),
        ('2024-03', 'Clothing',    16200.00)
    ) AS t(month, category, revenue)
""")

# Verify the table
print(conn.execute("SELECT * FROM sales").fetchdf())

In [ ]:
from langchain.tools import tool

# The tool closes over the `conn` variable — it has persistent access to the DB
@tool
def query_sales_database(sql: str) -> str:
    """Execute a SQL SELECT query against the sales database and return results as a table.
    Use this tool to answer questions about sales revenue, categories, or monthly trends.
    The database has one table: sales(month VARCHAR, category VARCHAR, revenue FLOAT).
    Input: a valid SQL SELECT statement. Only SELECT queries are allowed.
    Returns the query result as a formatted string or an error message."""
    # Guard against destructive queries
    if not sql.strip().upper().startswith("SELECT"):
        return "Error: Only SELECT statements are allowed."
    try:
        result = conn.execute(sql).fetchdf()
        if result.empty:
            return "Query returned no rows."
        # Format as a compact text table
        return result.to_string(index=False)
    except duckdb.Error as e:
        return f"SQL error: {str(e)}"

# Test it directly
print(query_sales_database.run("SELECT category, SUM(revenue) as total FROM sales GROUP BY category"))

**What just happened?**
- The `@tool` function closes over the DuckDB `conn` — the connection is shared across all tool calls in the session
- `SELECT`-only guard prevents the agent from accidentally mutating data
- `to_string(index=False)` returns a clean table the LLM can read and reason over
- **All errors are caught and returned as strings** — never let a tool raise an unhandled exception

## Step 4 · Register Tools in an AgentExecutor

Once tools are defined, wire them into an agent. We use `create_openai_tools_agent` — the modern approach that leverages OpenAI's native function-calling API instead of ReAct text parsing.

| Agent type | Mechanism | Best for |
|-----------|-----------|----------|
| `create_openai_tools_agent` | Native function calling | GPT-4o, reliable parsing |
| `create_react_agent` | Text ReAct loop | Any model, more verbose |
| `create_structured_chat_agent` | JSON in prompt | Claude, Mistral |

The `AgentExecutor` adds the **run loop**: call LLM → parse tool call → run tool → feed result back → repeat until no more tool calls.

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Set your key: os.environ['OPENAI_API_KEY'] = 'sk-...'
# For Colab: use the Secrets panel (key icon) to store OPENAI_API_KEY
# then: from google.colab import userdata; os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # temp=0 → deterministic tool selection

# Collect all three tools
tools = [get_wikipedia_summary, weather_tool, query_sales_database]

# Standard system prompt for tool-using agents
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to Wikipedia, weather data, and a sales database. "
               "Use tools whenever the question requires factual lookup or data querying. "
               "If a tool fails, acknowledge the error and explain what happened."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # required: stores intermediate tool results
])

agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,       # prints each tool call and result — essential for debugging
    max_iterations=5,   # prevents infinite loops
    handle_parsing_errors=True,  # recovers from malformed LLM output
)

print("Agent ready. Tools registered:", [t.name for t in tools])

**What just happened?**
- `MessagesPlaceholder("agent_scratchpad")` is mandatory — it's where intermediate tool call results accumulate during the run loop
- `temperature=0` is critical for tool-using agents — creativity degrades tool selection accuracy
- `max_iterations=5` is a safety net; a well-designed agent typically finishes in 2–3 iterations
- `handle_parsing_errors=True` lets the agent retry if the LLM returns malformed JSON

## Step 5 · Run a Query That Forces Both Tools

A good test of multi-tool routing asks a question that **requires two different tools** in one response. The agent should call both, not just pick one.

In [ ]:
# This question requires Wikipedia (what is DuckDB?) AND the sales DB (what's the top category?)
result = executor.invoke({
    "input": (
        "Give me a one-sentence description of DuckDB from Wikipedia, "
        "and also tell me which sales category had the highest total revenue in our database."
    )
})
print("\n=== Final answer ===")
print(result["output"])

In [ ]:
# Second query forces the weather tool specifically
result2 = executor.invoke({
    "input": "What's the weather like in Tokyo right now? Give me Celsius."
})
print("\n=== Final answer ===")
print(result2["output"])

**What just happened?**
- The first query triggered **two tool calls** in sequence — the agent composed the final answer from both results
- The second query cleanly routed to `get_weather` because the description matched the intent
- `verbose=True` output shows the exact tool name and input the agent chose — invaluable for debugging wrong tool selection
- If you see the agent pick the wrong tool, **improve the tool description**, not the prompt

## Step 6 · Tool Failure Recovery

Tools fail in production. A well-designed agent system **surfaces the error as information** rather than crashing. There are two layers:
1. **Within the tool** — catch exceptions, return an error string
2. **AgentExecutor** — `handle_parsing_errors=True` and `max_iterations` limit runaway loops

We'll simulate a tool that raises an exception and verify the agent responds gracefully.

In [ ]:
from langchain.tools import tool

@tool
def flaky_tool(ticker: str) -> str:
    """Look up the stock price for a given ticker symbol.
    Use this when asked about stock prices or equity valuations.
    Input: a stock ticker symbol like 'AAPL' or 'MSFT'.
    Returns the current price as a string."""
    # Simulate a network timeout or API outage
    raise ConnectionError(f"Stock API unreachable. Could not fetch price for {ticker}.")

# Build a second executor that includes the flaky tool
tools_with_flaky = tools + [flaky_tool]

agent2 = create_openai_tools_agent(llm=llm, tools=tools_with_flaky, prompt=prompt)
executor2 = AgentExecutor(
    agent=agent2,
    tools=tools_with_flaky,
    verbose=True,
    max_iterations=4,
    handle_parsing_errors=True,
)

# Query that forces the flaky tool
result3 = executor2.invoke({"input": "What is the current stock price of AAPL?"})
print("\n=== Final answer ===")
print(result3["output"])

**What just happened?**
- The tool raised a `ConnectionError` — LangChain caught it and **converted the exception message into a tool result string** that the agent can read
- The agent saw the error in its scratchpad and composed a response acknowledging the failure
- **Key insight:** By default, `AgentExecutor` catches tool exceptions and converts them to error strings when `handle_parsing_errors=True`. For explicit control, catch inside the tool and return a string yourself
- The agent didn't loop infinitely — `max_iterations` capped it

In [ ]:
# Challenge: Build a compound tool + agent
# Your task:
#   1. Write a @tool function `lookup_country_info(country: str) -> str` that:
#      - Fetches a 2-sentence Wikipedia summary for the country
#      - Appends the result of a DuckDB query for any sales data where category = country
#        (or a fallback message if no rows exist)
#      - Returns both pieces of info concatenated
#   2. Register it alongside weather_tool in a new AgentExecutor
#   3. Run a query: "Tell me about France and the weather there right now."
#      Verify the agent calls both lookup_country_info AND get_weather.

# Scaffold:
@tool
def lookup_country_info(country: str) -> str:
    """[Write a clear, specific description here — this is what the agent reads to decide when to call this tool.]"""
    # Step 1: Wikipedia summary
    wiki_result = ...  # call get_wikipedia_summary.run(...)

    # Step 2: DuckDB query (sales table may not have the country — handle gracefully)
    sql = f"SELECT * FROM sales WHERE LOWER(category) = LOWER('{country}') LIMIT 5"
    db_result = ...  # call query_sales_database.run(sql)

    return f"Wikipedia: {wiki_result}\n\nSales data: {db_result}"

# YOUR CODE HERE
# tools_v3 = [...]
# agent3 = ...
# executor3 = ...
# result4 = executor3.invoke({"input": "Tell me about France and the weather there right now."})
# print(result4["output"])

---
## Day 11 key concepts recap

| Concept | What to remember |
|---|---|
| `@tool` docstring | Becomes the tool description verbatim — this is the agent's only signal for when to call the tool |
| `StructuredTool` + Pydantic | Use for multi-arg tools; field `description` fields improve argument extraction |
| DuckDB integration | In-process SQL; return `to_string()` not raw DataFrames; guard with SELECT-only check |
| Error handling | Always catch inside the tool and return an error string — bare exceptions break the agent loop |
| `MessagesPlaceholder("agent_scratchpad")` | Mandatory in the prompt template for tool-using agents |
| `temperature=0` | Required for reliable tool selection — creativity degrades JSON argument extraction |

> **Tip:** Write clear, specific tool descriptions — the agent picks tools based entirely on their docstrings. Vague descriptions cause wrong tool selection.

---
## What's next
**Day 12** → Advanced RAG — MultiQueryRetriever, ContextualCompressionRetriever, and Ensemble Retrieval with BM25 + Chroma. You'll compare retrieval methods against labeled query-answer pairs to measure recall@4.

Mark Day 11 complete in your [tracker](../index.html).